# Solvent Diffusivity Physics-Informed GPR

This notebook builds single-task GPR models for experimental solvent diffusivity in polymers. It uses RDKit fingerprints for polymer repeat units and solvent molecules, then compares standard GPR with several physics-informed mean functions.

## Paper Reference

Nistane, J.; Datta, R.; Lee, Y. J.; Sahu, H.; Jang, S. S.; Lively, R.; Ramprasad, R. **Polymer design for solvent separations by integrating simulations, experiments and known physics via machine learning.** npj Computational Materials 11, 187 (2025).

Physics used here:

- Arrhenius temperature dependence: diffusivity increases with temperature through an activation-energy term.
- Solvent concentration/plasticization trend: diffusivity can increase with solvent uptake or concentration.
- Solvent-size trend: bulkier solvents generally diffuse more slowly.

The paper uses multitask and physics-enforced neural networks. This notebook keeps the first version as single-task GPR for interpretability and direct comparison.

## 1. Setup

In [ ]:
from pathlib import Path
import os
import sys

os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

NOTEBOOK_DIR = Path.cwd().resolve()


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "matgpr").exists():
            return candidate
        sibling = candidate / "matgpr"
        if (sibling / "pyproject.toml").exists() and (sibling / "matgpr").exists():
            return sibling
    raise FileNotFoundError("Could not find the local matgpr package root.")


PROJECT_ROOT = find_project_root(NOTEBOOK_DIR)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Notebook directory: {NOTEBOOK_DIR}")
print(f"Using package from: {PROJECT_ROOT}")

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
import torch
from gpytorch.utils.warnings import GPInputWarning
from scipy.stats import pearsonr
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler

from matgpr import PhysicsInformedMean, featurize_smiles, fit_gpytorch_gpr

RANDOM_STATE = 42
plt.style.use("default")
plt.rcParams.update(
    {
        "figure.dpi": 140,
        "savefig.dpi": 300,
        "font.size": 11,
        "axes.labelsize": 12,
        "axes.titlesize": 12,
        "legend.fontsize": 9,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": False,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    }
)
torch.set_num_threads(1)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=GPInputWarning)

## 2. Load and Filter Experimental Data

Only experimental diffusivity rows are used in this first notebook. Polymer repeat units must contain exactly two `[*]` atoms. The two ends are expanded into a cyclic trimer surrogate, connected using the dummy-end bond order, stripped of all `[*]` atoms, canonicalized with RDKit, and then fingerprinted.

In [ ]:
DATA_PATH = NOTEBOOK_DIR / "diffusivity_dataset.csv"
if not DATA_PATH.exists():
    DATA_PATH = PROJECT_ROOT / "examples" / "solvent_diffusivity" / "diffusivity_dataset.csv"

raw_data = pd.read_csv(DATA_PATH)
print(f"Raw rows: {len(raw_data):,}")
raw_data.head()

In [ ]:
data = raw_data.copy()
data = data[data["Experimental Selector"].eq(1)].copy()
data = data.rename(
    columns={
        "Polymer Canonical SMILES": "polymer_smiles",
        "Solvent Canonical SMILES": "solvent_smiles",
        "Temperature": "temperature_k",
        "Weight Fraction of Solvent": "weight_fraction_solvent",
        "Target Diffusivity Value(log10)": "log10_diffusivity",
    }
)
data = data.dropna(subset=["polymer_smiles", "solvent_smiles", "temperature_k", "weight_fraction_solvent", "log10_diffusivity"])
data = data[data["polymer_smiles"].astype(str).str.count(r"\[\*\]").eq(2)].copy()
data = data[data["weight_fraction_solvent"] > 0].reset_index(drop=True)

print(f"Experimental rows after strict polymer filter: {len(data):,}")
print(f"Unique polymers: {data['polymer_smiles'].nunique():,}")
print(f"Unique solvents: {data['solvent_smiles'].nunique():,}")
data.head()

## 3. RDKit Fingerprints

In [ ]:
FINGERPRINT_BITS = 128
FINGERPRINT_TYPE = "morgan+descriptors"

polymer_result = featurize_smiles(
    data["polymer_smiles"],
    smiles_type="polymer",
    fingerprint_type=FINGERPRINT_TYPE,
    n_bits=FINGERPRINT_BITS,
    column_prefix="polymer",
    errors="coerce",
)
solvent_result = featurize_smiles(
    data["solvent_smiles"],
    smiles_type="molecule",
    fingerprint_type=FINGERPRINT_TYPE,
    n_bits=FINGERPRINT_BITS,
    column_prefix="solvent",
    errors="coerce",
)

valid_mask = polymer_result.failed.empty and solvent_result.failed.empty
model_data = pd.concat(
    [
        data.reset_index(drop=True),
        polymer_result.canonical_smiles,
        solvent_result.canonical_smiles,
        polymer_result.features,
        solvent_result.features,
    ],
    axis=1,
)
model_data = model_data.dropna().reset_index(drop=True)
model_data["log_weight_fraction"] = np.log10(model_data["weight_fraction_solvent"].clip(lower=1e-8))
model_data["inverse_temperature_k"] = 1000.0 / model_data["temperature_k"]
model_data["solvent_molwt"] = model_data["solvent_desc_MolWt"]

print(f"Rows after RDKit featurization: {len(model_data):,}")
print(f"Feature columns: {polymer_result.features.shape[1] + solvent_result.features.shape[1]:,}")
model_data[["polymer_canonical_smiles", "solvent_canonical_smiles", "temperature_k", "weight_fraction_solvent", "log10_diffusivity"]].head()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12.4, 3.4))
axes[0].hist(model_data["log10_diffusivity"], bins=35, color="#2f6f8f", alpha=0.9)
axes[0].set_xlabel("log10 diffusivity")
axes[0].set_ylabel("Count")
axes[0].set_title("Target")
axes[1].scatter(model_data["temperature_k"], model_data["log10_diffusivity"], s=16, alpha=0.55, color="#b85c38")
axes[1].set_xlabel("Temperature (K)")
axes[1].set_ylabel("log10 diffusivity")
axes[1].set_title("Temperature trend")
axes[2].scatter(model_data["weight_fraction_solvent"], model_data["log10_diffusivity"], s=16, alpha=0.55, color="#4a7c59")
axes[2].set_xlabel("Weight fraction solvent")
axes[2].set_ylabel("log10 diffusivity")
axes[2].set_title("Concentration trend")
fig.tight_layout()

## 4. Physics-Informed Mean Functions

All models use the same GP covariance over RDKit fingerprints and condition features. Physics-informed variants differ only in the GP mean function.

In [ ]:
TARGET_COLUMN = "log10_diffusivity"
PHYSICS_FEATURES = ["temperature_k", "log_weight_fraction", "solvent_molwt"]
FINGERPRINT_COLUMNS = [
    column for column in model_data.columns
    if column.startswith("polymer_morgan_")
    or column.startswith("polymer_desc_")
    or column.startswith("solvent_morgan_")
    or column.startswith("solvent_desc_")
]
FEATURE_COLUMNS = FINGERPRINT_COLUMNS + PHYSICS_FEATURES

MODEL_CONFIGS = {
    "standard_gpr": {"label": "Standard GPR", "physics": "none"},
    "pi_arrhenius": {"label": "PI-GPR: Arrhenius", "physics": "arrhenius"},
    "pi_concentration": {"label": "PI-GPR: concentration", "physics": "concentration"},
    "pi_solvent_size": {"label": "PI-GPR: solvent size", "physics": "size"},
    "pi_combined": {"label": "PI-GPR: combined", "physics": "combined"},
}
MODEL_ORDER = list(MODEL_CONFIGS)
MODEL_LABELS = {key: value["label"] for key, value in MODEL_CONFIGS.items()}
MODEL_COLORS = {
    "standard_gpr": "#4c5a72",
    "pi_arrhenius": "#2f6f8f",
    "pi_concentration": "#4a7c59",
    "pi_solvent_size": "#8064a2",
    "pi_combined": "#b85c38",
}

TRAINING_PERCENTS = np.array([10, 30, 50, 70, 90, 100])
N_RANDOM_SPLITS = 10
TEST_SIZE = 0.20
TRAINING_ITER = 100
PRODUCTION_TRAINING_ITER = 200

In [ ]:
def arrhenius_mean(features, parameters):
    gas_constant = 8.314462618
    temperature = torch.clamp(features["temperature_k"], min=1.0)
    return parameters["log_d0"] - (parameters["activation_energy_kj_mol"] * 1000.0) / (
        np.log(10.0) * gas_constant * temperature
    )


def concentration_mean(features, parameters):
    return parameters["baseline"] + parameters["concentration_slope"] * features["log_weight_fraction"]


def size_mean(features, parameters):
    return parameters["baseline"] - parameters["size_penalty"] * torch.log10(torch.clamp(features["solvent_molwt"], min=1.0))


def combined_mean(features, parameters):
    gas_constant = 8.314462618
    temperature = torch.clamp(features["temperature_k"], min=1.0)
    arrhenius = parameters["log_d0"] - (parameters["activation_energy_kj_mol"] * 1000.0) / (
        np.log(10.0) * gas_constant * temperature
    )
    concentration = parameters["concentration_slope"] * features["log_weight_fraction"]
    size = parameters["size_penalty"] * torch.log10(torch.clamp(features["solvent_molwt"], min=1.0))
    return arrhenius + concentration - size


def build_physics_mean(model_key, feature_columns, scaler, y_train):
    physics = MODEL_CONFIGS[model_key]["physics"]
    if physics == "none":
        return None

    feature_means = dict(zip(feature_columns, scaler.mean_))
    feature_stds = dict(zip(feature_columns, scaler.scale_))
    y_train = np.asarray(y_train, dtype=float)
    common = {
        "feature_indices": {name: feature_columns.index(name) for name in PHYSICS_FEATURES},
        "feature_means": {name: feature_means[name] for name in PHYSICS_FEATURES},
        "feature_stds": {name: feature_stds[name] for name in PHYSICS_FEATURES},
    }

    if physics == "arrhenius":
        return PhysicsInformedMean(
            equation=arrhenius_mean,
            learnable_parameters={"log_d0": float(np.percentile(y_train, 75)), "activation_energy_kj_mol": 10.0},
            positive_parameters=("activation_energy_kj_mol",),
            **common,
        )
    if physics == "concentration":
        return PhysicsInformedMean(
            equation=concentration_mean,
            learnable_parameters={"baseline": float(np.median(y_train)), "concentration_slope": 0.5},
            positive_parameters=("concentration_slope",),
            **common,
        )
    if physics == "size":
        return PhysicsInformedMean(
            equation=size_mean,
            learnable_parameters={"baseline": float(np.median(y_train)), "size_penalty": 0.5},
            positive_parameters=("size_penalty",),
            **common,
        )
    if physics == "combined":
        return PhysicsInformedMean(
            equation=combined_mean,
            learnable_parameters={
                "log_d0": float(np.percentile(y_train, 75)),
                "activation_energy_kj_mol": 10.0,
                "concentration_slope": 0.5,
                "size_penalty": 0.5,
            },
            positive_parameters=("activation_energy_kj_mol", "concentration_slope", "size_penalty"),
            **common,
        )
    raise ValueError(f"Unknown physics model: {physics}")


def fit_model(model_key, train_frame, *, training_iter=TRAINING_ITER):
    scaler = StandardScaler()
    X_train = scaler.fit_transform(train_frame[FEATURE_COLUMNS])
    y_train = train_frame[TARGET_COLUMN].to_numpy(dtype=float)
    mean_module = build_physics_mean(model_key, FEATURE_COLUMNS, scaler, y_train)
    result = fit_gpytorch_gpr(
        X_train,
        y_train,
        kernel="matern",
        ard=False,
        mean_module=mean_module,
        lr=0.03,
        training_iter=training_iter,
        initial_noise=0.05,
        standardize_y=True,
        verbose=False,
    )
    return {"model_key": model_key, "scaler": scaler, "result": result}


def predict_model(fitted_model, frame, *, confidence_level=0.95):
    X = fitted_model["scaler"].transform(frame[FEATURE_COLUMNS])
    return fitted_model["result"].predict(X, confidence_level=confidence_level)

## 5. Learning Curves

In [ ]:
def regression_summary(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    r_value = pearsonr(y_true, y_pred).statistic if len(np.unique(y_true)) > 1 else np.nan
    return {
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "r2": float(r2_score(y_true, y_pred)),
        "r": float(r_value),
    }


def target_strata(frame, target_column, bins=8):
    return pd.qcut(frame[target_column], q=bins, labels=False, duplicates="drop")


def subset_training_pool(train_pool, target_column, training_percent, *, random_state):
    if training_percent >= 100:
        return train_pool.copy().reset_index(drop=True)
    train_size = training_percent / 100
    strata = target_strata(train_pool, target_column)
    try:
        subset, _ = train_test_split(
            train_pool,
            train_size=train_size,
            random_state=random_state,
            stratify=strata,
        )
    except ValueError:
        subset = train_pool.sample(frac=train_size, random_state=random_state)
    return subset.reset_index(drop=True)


def plot_learning_curves(learning_summary, model_order, model_labels, model_colors):
    fig, axes = plt.subplots(1, 2, figsize=(11.8, 4.1), sharex=True)
    for model_key in model_order:
        subset = learning_summary[learning_summary["model_key"] == model_key]
        color = model_colors.get(model_key)
        axes[0].errorbar(
            subset["training_percent"],
            subset["rmse_mean"],
            yerr=subset["rmse_std"],
            marker="o",
            linewidth=2.0,
            capsize=3,
            label=model_labels[model_key],
            color=color,
        )
        axes[1].errorbar(
            subset["training_percent"],
            subset["r2_mean"],
            yerr=subset["r2_std"],
            marker="o",
            linewidth=2.0,
            capsize=3,
            label=model_labels[model_key],
            color=color,
        )
    axes[0].set_ylabel("Held-out RMSE")
    axes[1].set_ylabel("Held-out R2")
    for ax in axes:
        ax.set_xlabel("Training pool used (%)")
        ax.set_xticks(TRAINING_PERCENTS)
        ax.legend(frameon=False)
    fig.tight_layout()
    return fig, axes

In [ ]:
learning_records = []

for repeat in range(N_RANDOM_SPLITS):
    split_seed = RANDOM_STATE + repeat
    train_pool, test_set = train_test_split(
        model_data,
        test_size=TEST_SIZE,
        random_state=split_seed,
        stratify=target_strata(model_data, TARGET_COLUMN),
    )
    train_pool = train_pool.reset_index(drop=True)
    test_set = test_set.reset_index(drop=True)

    for training_percent in TRAINING_PERCENTS:
        subset = subset_training_pool(train_pool, TARGET_COLUMN, int(training_percent), random_state=split_seed + int(training_percent) * 100)
        for model_key in MODEL_ORDER:
            fitted = fit_model(model_key, subset, training_iter=TRAINING_ITER)
            prediction = predict_model(fitted, test_set)
            metrics = regression_summary(test_set[TARGET_COLUMN], prediction.mean)
            learning_records.append(
                {
                    "repeat": repeat,
                    "training_percent": int(training_percent),
                    "n_train": len(subset),
                    "model_key": model_key,
                    "model": MODEL_LABELS[model_key],
                    **metrics,
                }
            )

learning_results = pd.DataFrame(learning_records)
learning_summary = (
    learning_results
    .groupby(["training_percent", "model_key", "model"], as_index=False)
    .agg(
        rmse_mean=("rmse", "mean"),
        rmse_std=("rmse", "std"),
        r2_mean=("r2", "mean"),
        r2_std=("r2", "std"),
        mae_mean=("mae", "mean"),
        mae_std=("mae", "std"),
    )
)
learning_summary.head()

In [ ]:
plot_learning_curves(learning_summary, MODEL_ORDER, MODEL_LABELS, MODEL_COLORS)

## 6. Select Candidate Physics Models

In [ ]:
SELECTION_PERCENT = 20
selection_table = (
    learning_summary[learning_summary["training_percent"] == SELECTION_PERCENT]
    .sort_values(["rmse_mean", "rmse_std"], ascending=[True, True])
    .reset_index(drop=True)
)
best_model_key = selection_table.loc[0, "model_key"]
best_model_label = MODEL_LABELS[best_model_key]
print(f"Selected low-data model: {best_model_label}")
selection_table[["model", "rmse_mean", "rmse_std", "r2_mean", "r2_std"]]

## 7. Low-Data Parity

In [ ]:
train_pool, test_set = train_test_split(
    model_data,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE + 999,
    stratify=target_strata(model_data, TARGET_COLUMN),
)
low_data_train = subset_training_pool(train_pool.reset_index(drop=True), TARGET_COLUMN, SELECTION_PERCENT, random_state=RANDOM_STATE + 123)

standard_model = fit_model("standard_gpr", low_data_train, training_iter=PRODUCTION_TRAINING_ITER)
best_model = fit_model(best_model_key, low_data_train, training_iter=PRODUCTION_TRAINING_ITER)
standard_pred = predict_model(standard_model, test_set)
best_pred = predict_model(best_model, test_set)
standard_metrics = regression_summary(test_set[TARGET_COLUMN], standard_pred.mean)
best_metrics = regression_summary(test_set[TARGET_COLUMN], best_pred.mean)

fig, axes = plt.subplots(1, 2, figsize=(10.6, 4.7), sharex=True, sharey=True)
for ax, label, pred, metrics, color in [
    (axes[0], "Standard GPR", standard_pred, standard_metrics, MODEL_COLORS["standard_gpr"]),
    (axes[1], best_model_label, best_pred, best_metrics, MODEL_COLORS.get(best_model_key, "#b85c38")),
]:
    ax.errorbar(test_set[TARGET_COLUMN], pred.mean, yerr=pred.std, fmt="o", ms=4, alpha=0.7, capsize=1.8, color=color)
    limits = [test_set[TARGET_COLUMN].min(), test_set[TARGET_COLUMN].max()]
    pad = 0.05 * (limits[1] - limits[0])
    limits = [limits[0] - pad, limits[1] + pad]
    ax.plot(limits, limits, "--", color="#2b2b2b", linewidth=1)
    ax.set_xlim(limits)
    ax.set_ylim(limits)
    ax.set_title(label)
    ax.set_xlabel("Observed log10 diffusivity")
    ax.text(0.04, 0.96, f"RMSE={metrics['rmse']:.3f}\\nR2={metrics['r2']:.3f}\\nr={metrics['r']:.3f}", transform=ax.transAxes, va="top", fontsize=9)
axes[0].set_ylabel("Predicted log10 diffusivity")
fig.tight_layout()

## 8. 90/10 Validation With 10-Fold Cross-Validation

The selected low-data model is evaluated with a conventional validation protocol before fitting the production model: 90 percent training data, 10 percent held-out test data, and 10-fold cross-validation inside the 90 percent training partition. The combined figure reports cross-validation statistics on the left and train/test parity with predictive uncertainty on the right.

In [ ]:
N_CV_SPLITS = 10
VALIDATION_TEST_SIZE = 0.10


def stratification_bins(n_rows, n_splits, max_bins=8):
    return max(2, min(max_bins, n_rows // max(n_splits * 2, 1)))


validation_train, validation_test = train_test_split(
    model_data,
    test_size=VALIDATION_TEST_SIZE,
    random_state=RANDOM_STATE + 900,
    stratify=target_strata(
        model_data,
        TARGET_COLUMN,
        bins=stratification_bins(len(model_data), N_CV_SPLITS),
    ),
)
validation_train = validation_train.reset_index(drop=True)
validation_test = validation_test.reset_index(drop=True)

cv_records = []
cv_strata = target_strata(
    validation_train,
    TARGET_COLUMN,
    bins=stratification_bins(len(validation_train), N_CV_SPLITS),
)
cv = StratifiedKFold(n_splits=N_CV_SPLITS, shuffle=True, random_state=RANDOM_STATE + 901)

for fold, (train_idx, val_idx) in enumerate(cv.split(validation_train, cv_strata), start=1):
    fold_train = validation_train.iloc[train_idx].reset_index(drop=True)
    fold_val = validation_train.iloc[val_idx].reset_index(drop=True)
    fitted = fit_model(best_model_key, fold_train, training_iter=TRAINING_ITER)

    train_pred = predict_model(fitted, fold_train)
    val_pred = predict_model(fitted, fold_val)
    train_metrics = regression_summary(fold_train[TARGET_COLUMN], train_pred.mean)
    val_metrics = regression_summary(fold_val[TARGET_COLUMN], val_pred.mean)

    cv_records.append({"fold": fold, "split": "CV train", **train_metrics})
    cv_records.append({"fold": fold, "split": "CV validation", **val_metrics})

cv_results = pd.DataFrame(cv_records)
cv_summary = (
    cv_results
    .groupby("split")
    .agg(
        rmse_mean=("rmse", "mean"),
        rmse_std=("rmse", "std"),
        r2_mean=("r2", "mean"),
        r2_std=("r2", "std"),
        mae_mean=("mae", "mean"),
        mae_std=("mae", "std"),
        r_mean=("r", "mean"),
        r_std=("r", "std"),
    )
    .reset_index()
)
cv_summary

In [ ]:
validation_model = fit_model(
    best_model_key,
    validation_train,
    training_iter=PRODUCTION_TRAINING_ITER,
)
validation_train_pred = predict_model(validation_model, validation_train)
validation_test_pred = predict_model(validation_model, validation_test)

validation_train_metrics = regression_summary(validation_train[TARGET_COLUMN], validation_train_pred.mean)
validation_test_metrics = regression_summary(validation_test[TARGET_COLUMN], validation_test_pred.mean)
validation_metrics = pd.DataFrame(
    [
        {"split": "Train", **validation_train_metrics},
        {"split": "Held-out test", **validation_test_metrics},
    ]
)


def format_metric(mean_value, std_value):
    return f"{mean_value:.3f} +/- {std_value:.3f}"


cv_table = cv_summary.copy()
for metric in ["rmse", "r2", "mae", "r"]:
    cv_table[metric.upper()] = [
        format_metric(row[f"{metric}_mean"], row[f"{metric}_std"])
        for _, row in cv_table.iterrows()
    ]
cv_table = cv_table[["split", "RMSE", "R2", "MAE", "R"]]

fig, axes = plt.subplots(1, 2, figsize=(12.2, 4.9), gridspec_kw={"width_ratios": [0.92, 1.35]})

axes[0].axis("off")
axes[0].set_title(f"{best_model_label}: 10-fold CV", pad=10)
table = axes[0].table(
    cellText=cv_table.values,
    colLabels=cv_table.columns,
    cellLoc="center",
    colLoc="center",
    loc="center",
)
table.auto_set_font_size(False)
table.set_fontsize(8.5)
table.scale(1.05, 1.45)
for (row, col), cell in table.get_celld().items():
    cell.set_linewidth(0.45)
    if row == 0:
        cell.set_facecolor("#e9eef2")
        cell.set_text_props(weight="bold")

ax = axes[1]
for frame, pred, split, color in [
    (validation_train, validation_train_pred, "Train (90%)", "#2f6f8f"),
    (validation_test, validation_test_pred, "Held-out test (10%)", "#b85c38"),
]:
    ax.errorbar(
        frame[TARGET_COLUMN],
        pred.mean,
        yerr=pred.std,
        fmt="o",
        ms=4.0,
        alpha=0.70,
        elinewidth=0.7,
        capsize=1.8,
        color=color,
        label=split,
    )

limits = [
    min(validation_train[TARGET_COLUMN].min(), validation_test[TARGET_COLUMN].min()),
    max(validation_train[TARGET_COLUMN].max(), validation_test[TARGET_COLUMN].max()),
]
pad = 0.05 * (limits[1] - limits[0]) if limits[1] > limits[0] else 1.0
limits = [limits[0] - pad, limits[1] + pad]
ax.plot(limits, limits, color="#2b2b2b", linewidth=1.1, linestyle="--")
ax.set_xlim(limits)
ax.set_ylim(limits)
ax.set_xlabel("Observed log10 diffusivity")
ax.set_ylabel("Predicted log10 diffusivity")
ax.set_title("90/10 train-test parity", pad=10)
ax.legend(frameon=False)

text = (
    f"Train RMSE={validation_train_metrics['rmse']:.3f}, R2={validation_train_metrics['r2']:.3f}, "
    f"r={validation_train_metrics['r']:.3f}\n"
    f"Test RMSE={validation_test_metrics['rmse']:.3f}, R2={validation_test_metrics['r2']:.3f}, "
    f"r={validation_test_metrics['r']:.3f}"
)
ax.text(0.04, 0.96, text, transform=ax.transAxes, va="top", ha="left", fontsize=9)
fig.tight_layout()

validation_metrics

## 9. Production Model on 100 Percent of the Data

The production model is refit with the selected model family using all available filtered data. This is the model used for the SHAP analysis below.

In [ ]:
production_model = fit_model(best_model_key, model_data, training_iter=PRODUCTION_TRAINING_ITER)
production_prediction = predict_model(production_model, model_data)
production_metrics = regression_summary(model_data[TARGET_COLUMN], production_prediction.mean)
print(f"Production model: {best_model_label}")
print(production_metrics)
mean_module = production_model["result"].model.mean_module
if hasattr(mean_module, "current_parameter_values"):
    print(mean_module.current_parameter_values())

## 10. SHAP Analysis for the Production Model

Permutation SHAP is run on a compact candidate feature set. The candidate set keeps the transport-condition physics features and the most target-correlated fingerprint or descriptor columns, while non-selected features are held at their median values. This keeps the analysis tractable for the larger solvent-diffusivity dataset.

In [ ]:
def select_shap_features(frame, feature_columns, target_column, *, always_include=(), max_features=30):
    selected = [feature for feature in always_include if feature in feature_columns]
    records = []
    for feature in feature_columns:
        values = pd.to_numeric(frame[feature], errors="coerce").to_numpy(dtype=float)
        target = frame[target_column].to_numpy(dtype=float)
        mask = np.isfinite(values) & np.isfinite(target)
        if mask.sum() < 3 or np.nanstd(values[mask]) == 0:
            score = 0.0
        else:
            score = abs(pearsonr(values[mask], target[mask]).statistic)
        records.append({"feature": feature, "score": score})
    ranked = pd.DataFrame(records).sort_values("score", ascending=False)
    for feature in ranked["feature"]:
        if feature not in selected:
            selected.append(feature)
        if len(selected) >= min(max_features, len(feature_columns)):
            break
    return selected, ranked


production_feature_columns = FEATURE_COLUMNS
SHAP_MAX_FEATURES = min(30, len(production_feature_columns))
SHAP_BACKGROUND_SIZE = min(40, len(model_data))
SHAP_EXPLAIN_SIZE = min(60, len(model_data))
SHAP_FEATURES, shap_feature_ranking = select_shap_features(
    model_data,
    production_feature_columns,
    TARGET_COLUMN,
    always_include=PHYSICS_FEATURES,
    max_features=SHAP_MAX_FEATURES,
)

reference_values = model_data[production_feature_columns].median(numeric_only=True)
shap_background = shap.sample(model_data[SHAP_FEATURES], SHAP_BACKGROUND_SIZE, random_state=RANDOM_STATE)
shap_explain = shap.sample(model_data[SHAP_FEATURES], SHAP_EXPLAIN_SIZE, random_state=RANDOM_STATE + 1)


def production_predict_for_shap(shap_frame):
    shap_frame = pd.DataFrame(shap_frame, columns=SHAP_FEATURES)
    full_frame = pd.DataFrame(
        np.repeat(reference_values.to_numpy()[None, :], len(shap_frame), axis=0),
        columns=production_feature_columns,
    )
    full_frame.loc[:, SHAP_FEATURES] = shap_frame.to_numpy()
    X_scaled = production_model["scaler"].transform(full_frame[production_feature_columns])
    return production_model["result"].predict(X_scaled, return_std=False).mean


explainer = shap.PermutationExplainer(production_predict_for_shap, shap_background)
shap_values = explainer(
    shap_explain,
    max_evals=2 * len(SHAP_FEATURES) + 1,
    batch_size=16,
)
shap_value_array = np.asarray(shap_values.values)
if shap_value_array.ndim == 3:
    shap_value_array = shap_value_array[:, :, 0]

shap_importance = (
    pd.DataFrame(
        {
            "feature": SHAP_FEATURES,
            "mean_abs_shap": np.abs(shap_value_array).mean(axis=0),
        }
    )
    .sort_values("mean_abs_shap", ascending=False)
    .reset_index(drop=True)
)
shap_importance.head(10).round(4)

In [ ]:
top_features = shap_importance.head(min(10, len(shap_importance)))["feature"].tolist()
bar_data = shap_importance.head(min(10, len(shap_importance))).iloc[::-1]
shap_predictions = production_predict_for_shap(shap_explain)

fig, axes = plt.subplots(1, 2, figsize=(12.2, 4.5), gridspec_kw={"width_ratios": [1.05, 1.0]})
axes[0].barh(bar_data["feature"], bar_data["mean_abs_shap"], color="#2f6f8f")
axes[0].set_xlabel("Mean absolute SHAP value (log10 diffusivity)")
axes[0].set_title("Top SHAP features")

feature = top_features[0]
feature_index = SHAP_FEATURES.index(feature)
scatter = axes[1].scatter(
    shap_explain[feature],
    shap_value_array[:, feature_index],
    c=shap_predictions,
    cmap="viridis",
    s=34,
    alpha=0.85,
    edgecolor="black",
    linewidth=0.25,
)
axes[1].axhline(0, color="#333333", linewidth=0.9, linestyle="--")
axes[1].set_xlabel(feature)
axes[1].set_ylabel("SHAP value (log10 diffusivity)")
axes[1].set_title(f"Dependence: {feature}")
colorbar = fig.colorbar(scatter, ax=axes[1])
colorbar.set_label("Predicted log10 diffusivity")
fig.tight_layout()